# Data Collection

## Objective

The goal of this section is to collect metadata about classical music, composers and time period from Open Opus API.
The collected information will be used as the foundation for building a classical music recommendation system.
Composer metadata will be later be combined with musical works and audio features to create meaningful representation of each piece.

In [27]:
import pandas as pd
import requests
import time

from pathlib import Path

### Get Composers Metadata from Open Opus API

First we connect to the Open Opus API and retrieve the list of available composers.

The request url for the API was found on the documentation of the API on GitHub.

List essential composers:GET /composer/list/rec.json

In [2]:
BASE_URL = 'https://api.openopus.org'
composer_url = f'{BASE_URL}/composer/list/rec.json'

In [3]:
response = requests.get(composer_url)

In [4]:
response.status_code

200

In [5]:
response.text[:500]

'{"status":{"version":"1.20.6","success":"true","source":"db","rows":77,"processingtime":0.0007059574127197266,"api":"Open Opus-dyn"},"request":{"type":"rec","item":"1"},"composers":[{"id":"149","name":"Adams","complete_name":"John Adams","birth":"1947-01-01","death":null,"epoch":"21st Century","portrait":"https:\\/\\/assets.openopus.org\\/portraits\\/74462091-1568084854.jpg"},{"id":"87","name":"Bach","complete_name":"Johann Sebastian Bach","birth":"1685-01-01","death":"1750-01-01","epoch":"Baroque",'

In [6]:
data = response.json()
data.keys()

dict_keys(['status', 'request', 'composers'])

In [7]:
data['composers'][:3]

[{'id': '149',
  'name': 'Adams',
  'complete_name': 'John Adams',
  'birth': '1947-01-01',
  'death': None,
  'epoch': '21st Century',
  'portrait': 'https://assets.openopus.org/portraits/74462091-1568084854.jpg'},
 {'id': '87',
  'name': 'Bach',
  'complete_name': 'Johann Sebastian Bach',
  'birth': '1685-01-01',
  'death': '1750-01-01',
  'epoch': 'Baroque',
  'portrait': 'https://assets.openopus.org/portraits/12091447-1568084857.jpg'},
 {'id': '125',
  'name': 'Bartók',
  'complete_name': 'Béla Bartók',
  'birth': '1881-01-01',
  'death': '1945-01-01',
  'epoch': '20th Century',
  'portrait': 'https://assets.openopus.org/portraits/70269699-1568084859.jpg'}]

### Convert API dictionary to pandas DataFrame

In [8]:
composer_df = pd.DataFrame(data['composers'])

In [9]:
composer_df.head()

,id,name,complete_name,birth,death,epoch,portrait
0,149,Adams,John Adams,1947-01-01,NaN,21st Century,https://assets.openopus.org/portraits/74462091...
1,87,Bach,Johann Sebastian Bach,1685-01-01,1750-01-01,Baroque,https://assets.openopus.org/portraits/12091447...
2,125,Bartók,Béla Bartók,1881-01-01,1945-01-01,20th Century,https://assets.openopus.org/portraits/70269699...
3,145,Beethoven,Ludwig van Beethoven,1770-01-01,1827-01-01,Early Romantic,https://assets.openopus.org/portraits/55910756...
4,210,Berg,Alban Berg,1885-01-01,1935-01-01,20th Century,https://assets.openopus.org/portraits/48656640...


In [10]:
composer_df.shape

(77, 7)

In [11]:
composer_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 77 entries, 0 to 76
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id             77 non-null     str  
 1   name           77 non-null     str  
 2   complete_name  77 non-null     str  
 3   birth          77 non-null     str  
 4   death          73 non-null     str  
 5   epoch          77 non-null     str  
 6   portrait       77 non-null     str  
dtypes: str(7)
memory usage: 4.3 KB


We collected information about 77 classical music composers and 7 features; id, name, complete_name, birth, death, epoch (time period) and link to the portrait of the composer.

### Save Composers Raw Metadata to data/raw/composers.csv

In [12]:
composer_df.to_csv(Path('../data/raw/composers.csv'), index=False)

### Get Each Composer's Work Metadata from Open Opus API

We use composer IDs collected in the previous section to retrieve the list of works for each composer and extract relevant metadata about each musical piece.

List works by composer ID: GET /work/list/composer/129/genre/all.json

In [33]:
works_list = []
i = 1
for composer_id in composer_df['id']:
    works_url = f'{BASE_URL}/work/list/composer/{composer_id}/genre/all.json'
    response = requests.get(works_url, timeout=20)
    print(i)
    if response.status_code == 200:
        data = response.json()
        works_by_composer = pd.DataFrame(data['works'])
        works_by_composer['composer_id'] = composer_id
        works_list.append(works_by_composer)
    i += 1
    time.sleep(0.2)

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77


In [37]:
works_df = pd.concat(works_list, ignore_index=True)

In [38]:
works_df.shape

(13014, 8)

In [39]:
works_df.head()

,title,subtitle,searchterms,popular,recommended,id,genre,composer_id
0,China Gates,,,1,1,16893,Keyboard,149
1,Doctor Atomic Symphony,,,0,1,16878,Orchestral,149
2,Harmonielehre,,,1,1,16875,Orchestral,149
3,Nixon in China,Opera,,1,1,16890,Stage,149
4,Short Ride in a Fast Machine,,,1,1,16896,Orchestral,149


### Save Works of Composers Raw Metadata to data/raw/works.csv

In [41]:
works_df.to_csv(Path('../data/raw/works.csv'), index=False)

### Merge Works and Composers by Costumer ID

In [52]:
works_composers_df = works_df.merge(
    composer_df,
    left_on='composer_id',
    right_on='id',
    how='left'
)

In [53]:
works_composers_df.head()

,title,subtitle,searchterms,popular,recommended,id_x,genre,composer_id,id_y,name,complete_name,birth,death,epoch,portrait
0,China Gates,,,1,1,16893,Keyboard,149,149,Adams,John Adams,1947-01-01,NaN,21st Century,https://assets.openopus.org/portraits/74462091...
1,Doctor Atomic Symphony,,,0,1,16878,Orchestral,149,149,Adams,John Adams,1947-01-01,NaN,21st Century,https://assets.openopus.org/portraits/74462091...
2,Harmonielehre,,,1,1,16875,Orchestral,149,149,Adams,John Adams,1947-01-01,NaN,21st Century,https://assets.openopus.org/portraits/74462091...
3,Nixon in China,Opera,,1,1,16890,Stage,149,149,Adams,John Adams,1947-01-01,NaN,21st Century,https://assets.openopus.org/portraits/74462091...
4,Short Ride in a Fast Machine,,,1,1,16896,Orchestral,149,149,Adams,John Adams,1947-01-01,NaN,21st Century,https://assets.openopus.org/portraits/74462091...


In [54]:
works_composers_df = works_composers_df.drop('id_y', axis=1)

In [55]:
works_composers_df = works_composers_df.rename(columns={'id_x': 'work_id'})

In [56]:
works_composers_df.head()

,title,subtitle,searchterms,popular,recommended,work_id,genre,composer_id,name,complete_name,birth,death,epoch,portrait
0,China Gates,,,1,1,16893,Keyboard,149,Adams,John Adams,1947-01-01,NaN,21st Century,https://assets.openopus.org/portraits/74462091...
1,Doctor Atomic Symphony,,,0,1,16878,Orchestral,149,Adams,John Adams,1947-01-01,NaN,21st Century,https://assets.openopus.org/portraits/74462091...
2,Harmonielehre,,,1,1,16875,Orchestral,149,Adams,John Adams,1947-01-01,NaN,21st Century,https://assets.openopus.org/portraits/74462091...
3,Nixon in China,Opera,,1,1,16890,Stage,149,Adams,John Adams,1947-01-01,NaN,21st Century,https://assets.openopus.org/portraits/74462091...
4,Short Ride in a Fast Machine,,,1,1,16896,Orchestral,149,Adams,John Adams,1947-01-01,NaN,21st Century,https://assets.openopus.org/portraits/74462091...


### Save The Merged Data to data/raw/works_with_composers.csv

In [57]:
works_composers_df.to_csv(Path('../data/raw/works_with_composers.csv'), index=False)

# Summary

In this notebook, the initial dataset for the Classical Music Recommendation System was collected using Open Opus API.

The data collection process included two main steps:
- Retrieving metadata for classical music composers, including composer IDs, names, birth and death dates and historical periods (epochs).
- Retrieving the complete list of available musical works for each composer using their unique composer IDs.

The collected composer and work datasets were merged to create a unified raw dataset containing information about both composers and their musical works. The objective of this notebook was solely to collect and preserve the raw data.

The resulting raw dataset was saved for use in the next stage of the project where data cleaning, feature engineering and preprocessing will be performed before exploratory data analysis and recommendation model development.